%% [markdown]
# CLIP inference parity — CPU / CUDA / Trainium
Runs openai CLIP's **own vendored definition** (`clip_neuron`) on every available device and checks
the outputs match. CPU is the reference; `cuda` is auto-skipped when absent (e.g. on the trn box);
`neuron` runs on the Trainium core. Pin a free core before running:
`NEURON_RT_VISIBLE_CORES=0 jupyter nbconvert --to notebook --execute 01_inference_parity.ipynb`.

In [1]:
# %%
import os
os.environ.setdefault("NEURON_RT_VISIBLE_CORES", "0")   # set before the Neuron runtime initializes
import sys
sys.path.insert(0, os.path.abspath("../src"))
import torch
import torch.nn.functional as F
import clip_reference as R

[W708 23:04:08.052474030 OperatorEntry.cpp:208] Warning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: aten::gather.out(Tensor self, int dim, Tensor index, *, bool sparse_grad=False, Tensor(a!) out) -> Tensor(a!)
    registered at /pytorch/build/aten/src/ATen/RegisterSchema.cpp:6
  dispatch key: PrivateUse1
  previous kernel: registered at /pytorch/build/aten/src/ATen/RegisterCPU_3.cpp:7637
       new kernel: registered at NeuronDispatcher.cpp:0 (function operator())


In [2]:
def devices():
    devs = ["cpu"]
    if torch.cuda.is_available():
        devs.append("cuda")
    try:
        import torch_neuronx  # noqa: F401
        devs.append("neuron")
    except Exception as e:
        print("neuron unavailable:", e)
    return devs

In [3]:
DEVICES = devices()
print("torch", torch.__version__, "| devices:", DEVICES)

torch 2.11.0+cpu | devices: ['cpu', 'neuron']


In [4]:
# %% [markdown]
# ## Run inference on each device
# %%
def run_inference(device):
    model = R.load(device)
    px, ids = R.build_inputs()
    with torch.no_grad():
        out = model(px.to(device), ids.to(device))
    if device == "neuron":
        import torch_neuronx; torch_neuronx.synchronize()
    return tuple(t.detach().float().cpu() for t in out)

In [5]:
results = {d: run_inference(d) for d in DEVICES}
for name, t in zip(R.OUTPUT_ORDER, results["cpu"]):
    print(f"cpu {name:18s} shape={tuple(t.shape)}")

cpu image_features     shape=(2, 512)
cpu text_features      shape=(2, 512)
cpu logits_per_image   shape=(2, 2)


In [6]:
# %% [markdown]
# ## Check every non-CPU device matches CPU (cosine + max-abs per output)
# %%
def cos(a, b): return F.cosine_similarity(a.flatten(), b.flatten(), dim=0).item()

In [7]:
ref = results["cpu"]
all_ok = True
for d in DEVICES:
    if d == "cpu":
        continue
    print(f"\n{d} vs cpu:")
    for name, a, b in zip(R.OUTPUT_ORDER, ref, results[d]):
        c = cos(a, b); mab = (a - b).abs().max().item()
        ok = c >= 0.99
        all_ok = all_ok and ok
        print(f"  {name:18s} cosine={c:.6f}  max-abs={mab:.3e}  {'OK' if ok else 'FAIL'}")


neuron vs cpu:
  image_features     cosine=1.000000  max-abs=1.240e-05  OK
  text_features      cosine=1.000000  max-abs=4.292e-06  OK
  logits_per_image   cosine=1.000000  max-abs=1.717e-05  OK


In [8]:
print("\nINFERENCE PARITY:", "PASS" if all_ok else "FAIL")
assert all_ok, "inference outputs diverged across devices"


INFERENCE PARITY: PASS
